In [2]:

from json import loads

from trafilatura import fetch_url, extract


# Collection of specific blog post/article URLs across different themes
urls = {
    "lifestyle": [
        "https://www.vogue.com/article/spring-2025-fashion-trends",
    ]
}


def scrape_url(url):
    """Scrape a single blog post/article URL"""
    try:
        downloaded = fetch_url(url=url)
        if not downloaded:
            return None

        result = extract(downloaded, output_format="json", include_comments=False, include_images=False,
                         include_links=False, with_metadata=True)

        if not result:
            return None

        result = loads(result)

        payload = {
            "image": result.get("image", ""),
            "url": result.get("source", url),
            "tags": result.get("tags", "").split(", ") if result.get("tags") else [],
            "title": result.get("title", ""),
            "description": result.get("description", ""),
            "content": result.get("raw_text", ""),
            "date": result.get("date", ""),
            "author": result.get("author", "")
        }
        return payload
    except Exception as e:
        print(f"Error scraping {url}: {str(e)}")
        return None

#
# # Scrape all articles
# scraped_data = []
# for theme, theme_urls in urls.items():
#     print(f"\nScraping {theme.upper()} articles...")
#     for url in theme_urls:
#         print(f"  Processing: {url}")
#         data = scrape_url(url)
#         if data:
#             data["theme"] = theme
#             scraped_data.append(data)
#
#         time.sleep(random.uniform(1, 3))
#
# print(f"\n✓ Successfully scraped {len(scraped_data)} articles")


In [6]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

URLS_PATH = "../../data/urls.txt"
CRAWLED_PATH = "../../data/crawled_pages.json"
MAX_WORKERS = 10        # concurrent fetches
SAVE_EVERY = 20         # flush to disk every N new pages

with open(URLS_PATH, "r") as f:
    url_list = json.load(f)

print(f"Loaded {len(url_list)} URLs from urls.txt")

if os.path.exists(CRAWLED_PATH):
    with open(CRAWLED_PATH, "r") as f:
        crawled_pages = json.load(f)
    print(f"Existing crawled_pages.json has {len(crawled_pages)} entries")
else:
    crawled_pages = {}
    print("No existing crawled_pages.json found, starting fresh")

# Filter out already-crawled URLs
to_crawl = [url for url in url_list if url not in crawled_pages]
skipped = len(url_list) - len(to_crawl)
print(f"Skipping {skipped} already-crawled URLs, {len(to_crawl)} remaining")

new_count = 0
failed = 0

def _save():
    with open(CRAWLED_PATH, "w") as f:
        json.dump(crawled_pages, f, indent=4, ensure_ascii=False)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(scrape_url, url): url for url in to_crawl}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Crawling URLs"):
        url = futures[future]
        try:
            data = future.result()
        except Exception as e:
            print(f"Error for {url}: {e}")
            data = None

        if data:
            crawled_pages[url] = data
            new_count += 1
            if new_count % SAVE_EVERY == 0:
                _save()
        else:
            failed += 1

# Final save
if new_count % SAVE_EVERY != 0:
    _save()

print(f"Done: {new_count} new pages scraped, {skipped} already existed, {failed} failed")
print(f"Total entries in crawled_pages.json: {len(crawled_pages)}")


Loaded 992 URLs from urls.txt
Existing crawled_pages.json has 945 entries
Skipping 945 already-crawled URLs, 47 remaining


Crawling URLs: 100%|██████████| 47/47 [00:00<00:00, 47.22it/s]

Done: 0 new pages scraped, 945 already existed, 47 failed
Total entries in crawled_pages.json: 945


In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

ADVERTISERS_PATH = "../../data/advertisers_list.json"
CRAWLED_ADVERTISERS_PATH = "../../data/crawled_advertisers.json"
MAX_WORKERS = 10
SAVE_EVERY = 20

with open(ADVERTISERS_PATH, "r") as f:
    advertisers_list = json.load(f)

# Build a list of (advertiser_id, website) tuples
advertiser_urls = [
    (adv["id"], adv["website"], adv.get("name", ""), adv.get("industry", ""))
    for adv in advertisers_list
    if adv.get("website")
]
print(f"Loaded {len(advertiser_urls)} advertiser URLs from advertisers_list.json")

if os.path.exists(CRAWLED_ADVERTISERS_PATH):
    with open(CRAWLED_ADVERTISERS_PATH, "r") as f:
        crawled_advertisers = json.load(f)
    print(f"Existing crawled_advertisers.json has {len(crawled_advertisers)} entries")
else:
    crawled_advertisers = {}
    print("No existing crawled_advertisers.json found, starting fresh")

# Filter out already-crawled URLs
to_crawl = [(aid, url, name, industry) for aid, url, name, industry in advertiser_urls if url not in crawled_advertisers]
skipped = len(advertiser_urls) - len(to_crawl)
print(f"Skipping {skipped} already-crawled URLs, {len(to_crawl)} remaining")

new_count = 0
failed = 0

def _save_advertisers():
    with open(CRAWLED_ADVERTISERS_PATH, "w") as f:
        json.dump(crawled_advertisers, f, indent=4, ensure_ascii=False)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(scrape_url, url): (aid, url, name, industry) for aid, url, name, industry in to_crawl}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Crawling advertiser URLs"):
        aid, url, name, industry = futures[future]
        try:
            data = future.result()
        except Exception as e:
            print(f"Error for {url}: {e}")
            data = None

        if data:
            data["advertiser_id"] = aid
            data["advertiser_name"] = name
            data["industry"] = industry
            crawled_advertisers[url] = data
            new_count += 1
            if new_count % SAVE_EVERY == 0:
                _save_advertisers()
        else:
            failed += 1

# Final save
if new_count % SAVE_EVERY != 0:
    _save_advertisers()

print(f"Done: {new_count} new advertiser pages scraped, {skipped} already existed, {failed} failed")
print(f"Total entries in crawled_advertisers.json: {len(crawled_advertisers)}")


Loaded 1000 advertiser URLs from advertisers_list.json
Existing crawled_advertisers.json has 723 entries
Skipping 759 already-crawled URLs, 241 remaining


Crawling advertiser URLs:  10%|▉         | 24/241 [00:31<04:42,  1.30s/it]


In [4]:
data = scrape_url("https://www.vogue.com/article/spring-2025-fashion-trends")

{'image': 'https://assets.vogue.com/photos/67c9df7ca1f72dd92fd6aee3/16:9/w_1280,c_limit/holding-rtw.png',
 'url': 'https://www.vogue.com/article/spring-2025-fashion-trends',
 'tags': ['shopping'],
 'title': 'The Top Spring/Summer 2025 Fashion Trends You Can Shop and Wear Now',
 'description': '',
 'content': 'All products featured on Vogue are independently selected by our editors. However, we may earn affiliate revenue on this article and commission when you buy something. While strong notions of femininity prevailed in the spring/summer collections, the spring/summer 2025 fashion trends we’re predicting to take over the fashion landscape are less ethereal and dreamy as they are practical and empowering. Rooted in a sense of “soft power,” which Vogue’s Laird Borelli-Persson connected to designers encouraging “an openness to a sense of wonder” in her analysis of the shows, these trends can be embraced in more ways than one. And key themes can already be incorporated into your wardrobe.

In [5]:
url = "https://www.vogue.com/article/9-outdated-fashion-pieces-vogue-editors-still-love"
downloaded = fetch_url(url=url)
result = extract(downloaded, output_format="json", include_comments=False, include_images=False, include_links=False,
                 with_metadata=True)

In [6]:
result = loads(result)

In [7]:
result

{'title': '9 Outdated Fashion Trends That Vogue Editors Still Love in 2025',
 'author': 'Christian Allaire',
 'hostname': 'vogue.com',
 'date': '2025-03-06',
 'fingerprint': '3e709b75d39f323d',
 'id': None,
 'license': None,
 'comments': '',
 'raw_text': 'Back in January, Timothée Chalamet stepped out in New York City wearing a serious throwback piece: An Alexander McQueen skull scarf. Having reached its peak in the late 2000s during the indie sleaze movement, that edgy skull-print scarf was worn by all the It girls at the time—from Lindsay Lohan to Mary-Kate Olsen—and is now considered a designer relic of the past. Outdated, even. But as Chalamet’s revival of it proves, who says something has to be out-of-style? When styled right, even the most forgotten fashion pieces can still feel fresh. The fashion moment got us thinking: Being trendy is seriously overrated, anyways. Sure, updated silhouettes and brand-new pieces can inject your wardrobe with a sense of newness—but shouldn’t our w

In [5]:
scraped_data[0]

NameError: name 'scraped_data' is not defined

KEYWORD AND NAMED ENTITY EXTRACTION

In [102]:
from keybert import KeyBERT

# KeyBERT will use "all-MiniLM-L6-v2" by default
kw_model = KeyBERT()

In [109]:


doc = scraped_data[0]["content"]

from itertools import chain

# Extract both unigram and bigram keywords
keywords = kw_model.extract_keywords(doc, keyphrase_ngram_range=(1, 1), top_n=20, stop_words="english")
keywords1 = kw_model.extract_keywords(doc, keyphrase_ngram_range=(1, 2), top_n=20, stop_words="english")

# Combine and keep only the highest scoring keyword occurrences
all_keywords = {}
for keyword, score in chain(keywords, keywords1):
    all_keywords[keyword] = max(score, all_keywords.get(keyword, 0))

print(all_keywords)

{'vogue': 0.608, 'fashion': 0.571, 'dresses': 0.4789, 'wardrobe': 0.4753, 'outerwear': 0.4378, 'dress': 0.4342, 'skirt': 0.4161, 'styles': 0.4123, 'skirts': 0.4115, 'dressing': 0.4005, 'tailored': 0.3575, 'trends': 0.3518, 'tailoring': 0.3516, 'shirts': 0.3478, 'wearing': 0.3417, 'designers': 0.3416, 'styled': 0.3396, 'wear': 0.3388, 'trend': 0.3333, 'denim': 0.33, 'vogue spring': 0.6652, 'wardrobe vogue': 0.66, 'fashion trends': 0.6556, 'fashion ongoing': 0.6273, 'featured vogue': 0.5965, 'vogue independently': 0.5949, 'predicting fashion': 0.5821, 'summertime dress': 0.5778, 'fashion mood': 0.5772, 'exemplifies fashion': 0.5712, '2025 fashion': 0.5522, 'fashion landscape': 0.5482, 'spring outerwear': 0.5328, 'vogue mark': 0.5192, 'fashion pov': 0.5129, 'skirts dresses': 0.5085, 'season outerwear': 0.5061, 'minimalist wardrobe': 0.505}


In [96]:
 !python -m spacy download en_core_web_sm

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [52]:
entities

[('Vogue', 'PERSON'),
 ('the spring/summer', 'DATE'),
 ('the spring/summer 2025', 'DATE'),
 ('Laird Borelli-Persson', 'PERSON'),
 ('Spring 2025', 'DATE'),
 ('2,490', 'MONEY'),
 ('The Bohemian Blouse', 'ORG'),
 ('Banana Republic', 'GPE'),
 ('100', 'MONEY'),
 ('Prada', 'ORG'),
 ('2,950', 'MONEY'),
 ('The Vintage-Inspired Print: Dries Van Noten', 'ORG'),
 ('620', 'MONEY'),
 ('The Checked Top: Simkhai Calliope', 'WORK_OF_ART'),
 ('495', 'MONEY'),
 ('The Frankie Shop Bea', 'ORG'),
 ('345', 'MONEY'),
 ('Diotima Darliston', 'PERSON'),
 ('1,095', 'MONEY'),
 ('The New Proportions Mini', 'ORG'),
 ('Alaïa', 'PERSON'),
 ('3,900', 'MONEY'),
 ('148', 'CARDINAL'),
 ('1,798', 'MONEY'),
 ('Aligne Omari', 'PERSON'),
 ('90', 'MONEY'),
 ('Tory Burch', 'PERSON'),
 ('2,898', 'MONEY'),
 ('the season ahead', 'DATE'),
 ('Bottega', 'PERSON'),
 ('Suede', 'ORG'),
 ('last fall', 'DATE'),
 ('first', 'ORDINAL'),
 ('spring', 'DATE'),
 ('this spring', 'DATE'),
 ('Van Noten', 'PERSON'),
 ('Gucci', 'PERSON'),
 ('Miu Miu

TOPIC CLASSIFICATION WITH ZERO-SHOT LEARNING

In [14]:

import json

with open("../data/iab_content_taxonomy.json", "r") as f:
    iab_taxonomy = json.load(f)

In [2]:
import os
from typing import List, Dict, Any, Tuple
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sentence_transformers import SentenceTransformer

# 0) Runtime setup
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")


def _device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


DEVICE = _device()
DTYPE = torch.float16 if DEVICE.type in {"cuda", "mps"} else torch.float32

# 1) Choose model (DeBERTa: better accuracy/512t; BART: 1024t -> fewer chunks)
MODEL_ID = "MoritzLaurer/deberta-v3-large-zeroshot-v2.0"  # or "facebook/bart-large-mnli"

# Build once
_zs_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
_zs_tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
_zs_model.to(DEVICE).eval()
classifier = pipeline(
    "zero-shot-classification",
    model=_zs_model,
    tokenizer=_zs_tok,
    device=0 if DEVICE.type == "cuda" else (-1 if DEVICE.type == "cpu" else DEVICE),
    # pipeline accepts torch.device in recent HF
    batch_size=8
)
tokenizer = classifier.tokenizer

# Fast embedder to shortlist labels per tier
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

Device set to use mps


In [24]:


# 2) Chunking with low overlap (fewer chunks)
def chunk_text_by_tokens(text: str, max_tokens: int | None = None, overlap: int = 32) -> List[str]:
    limit = tokenizer.model_max_length if max_tokens is None else min(max_tokens, tokenizer.model_max_length)
    usable = max(64, limit - 16)  # leave space for specials
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= usable:
        return [text]
    stride = max(1, usable - overlap)
    chunks = []
    for start in range(0, len(ids), stride):
        chunk_ids = ids[start:start + usable]
        if not chunk_ids:
            break
        chunks.append(tokenizer.decode(chunk_ids, skip_special_tokens=True))
        if start + usable >= len(ids):
            break
    return chunks


# 3) Shortlist labels with embeddings to cut NLI work by ~10x
def shortlist_labels(text_chunks: List[str], labels: List[str], top_k: int = 8) -> List[str]:
    if len(labels) <= top_k:
        return labels
    # Encode once per tier
    chunk_vecs = embedder.encode(text_chunks, batch_size=32, normalize_embeddings=True, convert_to_numpy=True)
    label_vecs = embedder.encode(labels, batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    sims = label_vecs @ chunk_vecs.T  # [L, C]
    per_label = sims.max(axis=1)  # best-chunk similarity for each label
    idx = np.argsort(-per_label)[:top_k]
    return [labels[i] for i in idx]


# 4) Score a single tier with chunking + label shortlisting + aggregation
def score_tier(
        text: str,
        candidate_labels: List[str],
        multi_label: bool,
        hypothesis_template: str,
        aggregate: str = "mean",
        batch_size: int = 8,
        max_tokens: int | None = None,
        overlap: int = 32,
        shortlist_top_k: int = 8
) -> Dict[str, float]:
    chunks = chunk_text_by_tokens(text, max_tokens=max_tokens, overlap=overlap)
    labels = shortlist_labels(chunks, candidate_labels, top_k=min(shortlist_top_k, len(candidate_labels)))
    # Batch over all chunks in one call
    results = classifier(
        sequences=chunks if len(chunks) > 1 else chunks[0],
        candidate_labels=labels,
        multi_label=multi_label,
        hypothesis_template=hypothesis_template,
        batch_size=batch_size
    )
    if isinstance(results, dict):
        results = [results]

    per_label_scores: Dict[str, List[float]] = {lab: [] for lab in labels}
    for out in results:
        for lab, s in zip(out["labels"], out["scores"]):
            per_label_scores[lab].append(float(s))

    def reduce(vals: List[float]) -> float:
        if not vals:
            return 0.0
        return max(vals) if aggregate == "max" else (sum(vals) / len(vals))

    # Produce scores only for shortlisted labels; others are zero
    scores = {lab: reduce(v) for lab, v in per_label_scores.items()}
    for lab in candidate_labels:
        scores.setdefault(lab, 0.0)
    return scores

In [25]:
def hierarchical_zero_shot(
        text: str,
        hierarchy: List[Dict[str, Any]],
        multi_label_per_tier: bool = True,
        tier_threshold: float = 0.5,
        tier_top_k: int = 2,
        hypothesis_template: str = "This text is about {}.",
        score_aggregate: str = "mean",
        max_tokens: int | None = None,
        overlap: int = 32,
        batch_size: int = 8,
        shortlist_top_k: int = 8,
        return_top_paths: int = 5
) -> List[List[Dict[str, Any]]]:
    """
    Hierarchical zero-shot classification that returns complete paths through the taxonomy.

    Returns:
        List of paths, where each path is a list of dictionaries containing:
        - category: The category name
        - iab_id: The IAB taxonomy ID
        - tier: The tier level (1, 2, 3, etc.)
        - score: The classification score for this tier
    """
    # Store branches with full tier information: (nodes, path_with_details, cum_score)
    branches: List[Tuple[List[Dict[str, Any]], List[Dict[str, Any]], float]] = [(hierarchy, [], 1.0)]
    completed: List[Tuple[List[Dict[str, Any]], float]] = []

    while branches:
        next_branches: List[Tuple[List[Dict[str, Any]], List[Dict[str, Any]], float]] = []
        for nodes, path_details, cum_score in branches:
            if not nodes:
                completed.append((path_details, cum_score))
                continue

            labels = [n.get("name", "") for n in nodes if n.get("name")]
            if not labels:
                completed.append((path_details, cum_score))
                continue

            # Determine current tier level
            current_tier = len(path_details) + 1

            scores = score_tier(
                text=text,
                candidate_labels=labels,
                multi_label=multi_label_per_tier,
                hypothesis_template=hypothesis_template,
                aggregate=score_aggregate,
                batch_size=batch_size,
                max_tokens=max_tokens,
                overlap=overlap,
                shortlist_top_k=shortlist_top_k
            )

            ranked = sorted(((lab, scores.get(lab, 0.0)) for lab in labels), key=lambda x: x[1], reverse=True)
            if multi_label_per_tier:
                chosen = [(lab, s) for lab, s in ranked if s >= tier_threshold][:tier_top_k]
            else:
                lab, s = ranked[0]
                chosen = [(lab, s)] if s >= tier_threshold else []

            if not chosen:
                completed.append((path_details, cum_score))
                continue

            for lab, s in chosen:
                node = next((n for n in nodes if n.get("name") == lab), None)
                if node:
                    node_id = node.get("id", "")
                    child_nodes = node.get("children", [])

                    # Create tier entry with all required information
                    tier_entry = {
                        "category": lab,
                        "iab_id": node_id,
                        "tier": current_tier,
                        "score": s
                    }

                    # Create new path with this tier added
                    new_path_details = path_details + [tier_entry]
                    next_branches.append((child_nodes, new_path_details, cum_score * s))

        if not next_branches:
            break
        branches = next_branches

    # Add any remaining branches to completed
    for nodes, path_details, cum_score in branches:
        completed.append((path_details, cum_score))

    # Sort by cumulative score (descending)
    completed = sorted(completed, key=lambda x: x[1], reverse=True)

    # Return top paths, each path is a list of tier dictionaries
    result = []
    seen_leaf_ids = set()

    for path_details, cum_score in completed:
        if not path_details:
            continue

        # Use the last tier's iab_id as the leaf identifier to avoid duplicates
        leaf_id = path_details[-1]["iab_id"]
        if leaf_id in seen_leaf_ids:
            continue

        result.append(path_details)
        seen_leaf_ids.add(leaf_id)

        if len(result) >= return_top_paths:
            break

    return result

In [26]:
paths = hierarchical_zero_shot(
    text=result["raw_text"],
    hierarchy=iab_taxonomy,
    multi_label_per_tier=True,
    tier_threshold=0.2,
    tier_top_k=2,
    hypothesis_template="The topic of this text is {}.",
    score_aggregate="mean",  # or "max"
    max_tokens=300,  # use model limit
    overlap=96,
    batch_size=8,
    return_top_paths=5
)
print(paths)

[[{'category': 'Shopping', 'iab_id': '473', 'tier': 1, 'score': 0.850313534339269}], [{'category': 'Style & Fashion', 'iab_id': '552', 'tier': 1, 'score': 0.9989955557717217}, {'category': 'Fashion Trends', 'iab_id': '577', 'tier': 2, 'score': 0.7985988954703013}], [{'category': 'Style & Fashion', 'iab_id': '552', 'tier': 1, 'score': 0.9989955557717217}, {'category': "Women's Fashion", 'iab_id': '560', 'tier': 2, 'score': 0.7913953156934844}, {'category': "Women's Clothing", 'iab_id': '566', 'tier': 3, 'score': 0.8073450467652745}, {'category': "Women's Casual Wear", 'iab_id': '568', 'tier': 4, 'score': 0.36026095712764394}], [{'category': 'Style & Fashion', 'iab_id': '552', 'tier': 1, 'score': 0.9989955557717217}, {'category': "Women's Fashion", 'iab_id': '560', 'tier': 2, 'score': 0.7913953156934844}, {'category': "Women's Clothing", 'iab_id': '566', 'tier': 3, 'score': 0.8073450467652745}, {'category': "Women's Outerwear", 'iab_id': '571', 'tier': 4, 'score': 0.35540346674517626}], 

[{'category': 'Shopping', 'iab_id': '473', 'score': 0.850313534339269}, {'category': 'Style & Fashion > Fashion Trends', 'iab_id': '577', 'score': 0.7977967474190367}, {'category': "Style & Fashion > Women's Fashion > Women's Clothing > Women's Casual Wear", 'iab_id': '568', 'score': 0.2299500006539728}, {'category': "Style & Fashion > Women's Fashion > Women's Clothing > Women's Outerwear", 'iab_id': '571', 'score': 0.22684952613813636}, {'category': "Style & Fashion > Women's Fashion > Women's Accessories > Women's Handbags and Wallets", 'iab_id': '563', 'score': 0.06246488952733527}]


In [12]:
testing = [{'category': 'Shopping', 'iab_id': '473', 'score': 0.850313534339269},
           {'category': 'Style & Fashion > Fashion Trends', 'iab_id': '577', 'score': 0.7977967474190367},
           {'category': "Style & Fashion > Women's Fashion > Women's Clothing > Women's Casual Wear", 'iab_id': '568',
            'score': 0.2299500006539728},
           {'category': "Style & Fashion > Women's Fashion > Women's Clothing > Women's Outerwear", 'iab_id': '571',
            'score': 0.22684952613813636},
           {'category': "Style & Fashion > Women's Fashion > Women's Accessories > Women's Handbags and Wallets",
            'iab_id': '563', 'score': 0.06246488952733527}]

{'title': '9 Outdated Fashion Trends That Vogue Editors Still Love in 2025',
 'author': 'Christian Allaire',
 'hostname': 'vogue.com',
 'date': '2025-03-06',
 'fingerprint': '3e709b75d39f323d',
 'id': None,
 'license': None,
 'comments': '',
 'raw_text': 'All products featured on Vogue are independently selected by our editors. However, we may receive compensation from retailers and/or from purchases of products through these links. Back in January, Timothée Chalamet stepped out in New York City wearing a serious throwback piece: An Alexander McQueen skull scarf. Having reached its peak in the late 2000s during the indie sleaze movement, that edgy skull-print scarf was worn by all the It girls at the time—from Lindsay Lohan to Mary-Kate Olsen—and is now considered a designer relic of the past. Outdated, even. But as Chalamet’s revival of it proves, who says something has to be out-of-style? When styled right, even the most forgotten fashion pieces can still feel fresh. The fashion mome

EMBEDDING AND CHUNKING FOR SEMANTIC

In [34]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Load model once (not every call)
model = SentenceTransformer('all-MiniLM-L6-v2')


def sent_tokenize(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]


def semantic_chunker(content: str, breakpoint_percentile_threshold: float = 90.0) -> list[str]:
    """
    Splits text segments into semantically cohesive chunks.
    Breaks where similarity between adjacent segments falls below the given percentile.

    Args:
        content: Text to be split.
        breakpoint_percentile_threshold: Percentile (0-100) of similarity scores.
                                         Higher values = fewer splits (stricter).

    Returns:
        List of text chunks.
    """
    texts: list[str] = sent_tokenize(content)
    if not texts:
        return []
    if len(texts) == 1:
        return texts

    # Compute embeddings
    embeddings = model.encode(texts, convert_to_tensor=True)

    # Compute similarity between consecutive sentences
    similarities = util.cos_sim(embeddings[:-1], embeddings[1:]).diagonal().cpu().numpy()

    # Compute similarity threshold
    threshold = np.percentile(similarities, breakpoint_percentile_threshold)

    # Determine where to split
    split_indices = [i for i, sim in enumerate(similarities) if sim < threshold]

    # Build chunks
    chunks = []
    start = 0
    for idx in split_indices:
        chunks.append(" ".join(texts[start:idx + 1]))
        start = idx + 1

    if start < len(texts):
        chunks.append(" ".join(texts[start:]))

    return chunks

In [87]:
chunks = semantic_chunker(content=result["title"] + " . " + result["raw_text"], breakpoint_percentile_threshold=50.0)
print(len(chunks))

40


In [88]:
chunks

['9 Outdated Fashion Trends That Vogue Editors Still Love in 2025 . All products featured on Vogue are independently selected by our editors.',
 'However, we may receive compensation from retailers and/or from purchases of products through these links.',
 'Back in January, Timothée Chalamet stepped out in New York City wearing a serious throwback piece: An Alexander McQueen skull scarf. Having reached its peak in the late 2000s during the indie sleaze movement, that edgy skull-print scarf was worn by all the It girls at the time—from Lindsay Lohan to Mary-Kate Olsen—and is now considered a designer relic of the past.',
 'Outdated, even. But as Chalamet’s revival of it proves, who says something has to be out-of-style? When styled right, even the most forgotten fashion pieces can still feel fresh. The fashion moment got us thinking: Being trendy is seriously overrated, anyways. Sure, updated silhouettes and brand-new pieces can inject your wardrobe with a sense of newness—but shouldn’t 

In [91]:
from sentence_transformers import SentenceTransformer, util


def sent_tokenize(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]


def semantic_chunk(text, similarity_threshold=0.4, max_chunk_sentences=5):
    # Load a small, efficient embedding model
    model = SentenceTransformer('all-MiniLM-L6-v2')

    sentences = sent_tokenize(text)

    # Step 2: Compute embeddings
    embeddings = model.encode(sentences, convert_to_tensor=True)

    chunks = []
    current_chunk = [sentences[0]]
    current_sims = []

    for i in range(1, len(sentences)):
        sim = util.cos_sim(embeddings[i - 1], embeddings[i]).item()
        current_sims.append(sim)

        # Compute running average similarity in current chunk
        avg_sim = sum(current_sims) / len(current_sims)

        if avg_sim < similarity_threshold or len(current_chunk) >= max_chunk_sentences:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
            current_sims = []
        else:
            current_chunk.append(sentences[i])

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


# Example Usage
text = """
Artificial intelligence is transforming industries around the world.
Machine learning and deep learning are key components of AI.
They are used in healthcare, finance, and transportation.
On the other hand, there are concerns about bias and fairness.
Regulators are trying to ensure responsible AI development.
"""

chunks = semantic_chunk(text)
for i, c in enumerate(chunks, 1):
    print(f"Chunk {i}:\n{c}\n")

Chunk 1:
Artificial intelligence is transforming industries around the world. Machine learning and deep learning are key components of AI.

Chunk 2:
They are used in healthcare, finance, and transportation.

Chunk 3:
On the other hand, there are concerns about bias and fairness.

Chunk 4:
Regulators are trying to ensure responsible AI development.



In [40]:
ans = embedder.encode(chunks, convert_to_numpy=True)

In [42]:
len(ans[30])

384

In [63]:
def generate_hash_and_url(url: str) -> tuple[str, str]:
    canonical_url = canonicalize_url(url)
    print("Canonical URL: ", canonical_url)

    url_hash = hashlib.sha256(canonical_url.encode('utf-8')).hexdigest()
    return url_hash, canonical_url

In [67]:
generate_hash_and_url('https://www.vogue.com/article/9-outdated-fashion-pieces-vogue-editors-still-love')


Canonical URL:  https://www.vogue.com/article/9-outdated-fashion-pieces-vogue-editors-still-love


('8e052c7004d0f4683be026d7f1f5ea799e00437f455765aac68f7b3eb01e126e',
 'https://www.vogue.com/article/9-outdated-fashion-pieces-vogue-editors-still-love')

In [72]:
from typing import Literal
import hashlib
import urllib.parse
import urllib


def canonical_url(url: str) -> Literal[b""]:
    parsed = urllib.parse.urlparse(url)
    normalized = parsed._replace(
        scheme=parsed.scheme.lower(),
        netloc=parsed.netloc.lower(),
        path=parsed.path.rstrip('/')
    )
    return urllib.parse.urlunparse(normalized)


def url_id(url: str) -> str:
    normalized = canonical_url(url)
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

In [73]:
url_id('https://www.vogue.com/article/9-outdated-fashion-pieces-vogue-editors-still-love')


'8e052c7004d0f4683be026d7f1f5ea799e00437f455765aac68f7b3eb01e126e'

get the complete path and have a the probability of that happpeing, so multiply the P(T1)X P(T2).. P(TN)

In [5]:
from openai import OpenAI
from python.prompts.prompt import NER_EXTRACTION_PROMPT
import json
from typing import List, Dict

client = OpenAI(
    api_key="",)


def get_named_entities(content: str) -> List[Dict[str, str]]:
    response = client.chat.completions.create(
        model="gpt-5",
        messages=[
            {"role": "system", "content": NER_EXTRACTION_PROMPT},
            {"role": "user", "content": content}
        ]
    )
    try:
        raw = response.choices[0].message.content
    except Exception:
        try:
            raw = response.choices[0]["message"]["content"]
        except Exception:
            raw = response.choices[0].get("text", "") if response.choices else ""
    try:
        return json.loads(raw)
    except Exception:
        return []


# Example


ModuleNotFoundError: No module named 'python'

In [6]:
text = "The Top Spring/Summer 2025 Fashion Trends You Can Shop and Wear Now..All products featured on Vogue are independently selected by our editors. However, we may earn affiliate revenue on this article and commission when you buy something. While strong notions of femininity prevailed in the spring/summer collections, the spring/summer 2025 fashion trends we’re predicting to take over the fashion landscape are less ethereal and dreamy as they are practical and empowering. Rooted in a sense of “soft power,” which Vogue’s Laird Borelli-Persson connected to designers encouraging “an openness to a sense of wonder” in her analysis of the shows, these trends can be embraced in more ways than one. And key themes can already be incorporated into your wardrobe. Vogue’s Top Spring 2025 Fashion Trends: The Summertime Dress: Proenza Schouler Yves stripe fringed knit dress, $2,490 The Bohemian Blouse: Banana Republic chiffon twist-neck top, $100 The Preppy Pleated Skirt: Prada rush stitch skirt, $2,950 The Vintage-Inspired Print: Dries Van Noten printed midi skirt, $620 The Checked Top: Simkhai Calliope tailored vest, $495 The New Business Blazer: The Frankie Shop Bea oversized cady blazer, $345 The Crafty Minimalism Skirt: Diotima Darliston skirt, $1,095 The New Proportions Mini: Alaïa cotton poplin mini dress, $3,900 The Romantic Floral Dress: Lafayette 148 New York Gesture handkerchief dress, $1,798 The Coastal Stripe: Aligne Omari striped shorts, $90 The New-Season Suede: Tory Burch suede bomber jacket, $2,898 Consider spring’s outerwear stories, which reimagine lightweight jackets of all types, particularly sporty styles for the season ahead. For inspiration on how to wear these jackets from now, look no further than Bottega, where an elasticated windbreaker was styled with fluid trousers. A zip-up à la Gucci works equally well here—just throw an overcoat on top. Suede will continue to trend as well, so if you invested last fall, keep wearing it! Dark chocolate brown styles pair particularly nicely with spring’s softer color palette of wispy pinks and pale yellows. As for the print that reigned supreme on the spring/summer runways? It wasn’t a floral or an animal but plaid, and its first cousin, gingham. The autumnal pattern was already a favorite last fall, and it seems like fashion wasn’t done with it for spring, rendering it in softer colors and shapes. If your wardrobe already has a spirited amount of checks, look to the runways for ideas on how to print clash—another major theme this spring that was put forward at Dries Van Noten, Gucci, and Miu Miu, where high-impact, retro-inspired prints were piled on in an artfully chaotic manner. (If it looks wrong, it’s probably right.) These and more key spring/summer 2025 fashion trends to know and shop now as spring deliveries land in stores. Femininity Unraveled Soft pastels brought romance and airiness to the runways, but the color palette didn’t feel overly feminine. Even minimalists will be drawn to this season’s wispy pinks and pale yellows, which look great against wardrobe neutrals like grey, navy, and even chocolate brown suede. Summertime Prep Inspired by sunny weekends on the coast, but really for any kind of weather or locale, the abundance of seaside-inspired pieces offer imaginative dressing solutions that are both practical and seasonally appropriate. Key pieces include anything with a sailor stripe, buoyant cotton skirts and dresses, and wind-resistant jackets—it does get blustery out there! Spring Outerwear Stories A sporty sensibility carried over from fashion’s Olympics fervor pervaded the spring/summer 2025 runways. Sabato de Sarno showed various fitted zip-up jackets at Gucci, while Bottega and Miu Miu reimagined the wind breaker—Matthieu Blazy notably lined his in plaid, a carryover from fall we’ll get into next. But other kinds of lightweight outerwear were also introduced. At Prada, Miuccia and Raf Simons showed a suede style, proof that fashion’s ongoing obsession with the texture isn’t going anywhere. A cropped leather cape made an appearance at Loewe, upleveling the boho staple we were reacquainted with last spring. Investing in next-season outerwear now doesn’t mean you have to wait until March to wear your pieces—layer similar styles from Gucci and Loewe under heavier coats to maximize their cost per wear. Checks and Balances Plaid may be synonymous with fall—where it indeed saw a resurgence this season—but its presence in the spring collections takes on a softer tune. Borelli-Persson described these checks best as “Nirvana meets Country Living… for any season or type of weather” in her trend report. The Row’s ‘Tavishina’ coat in look 29 exemplifies the fashion mood, with an energizing combination of beige, black, and baby blue squares that was styled with pinstripe pants and a plain white button-up for a harmonious, minimalist-approved take on print mixing. Blazy meanwhile played with a classic color scheme of browns, greys, and white, but showed it as an oversized shacket meets coat hybrid. At Tod’s checks were playfully introduced once more, worn inside out as a full ensemble. In New York, Daniella Kallmeyer furthered the artful drape with a fluid blouse, while Acne Studios dabbled in the art of the clash (nails included!) and incorporated another theme seen in the resort 2025 trends: the bubble hem. This ’90s-inspired look can be achieved now with printed pieces from the likes of Massimo Dutti, Ralph Lauren, or Zara—and styled with pattern opposites for a true runway-coded look. Let’s Go Thrifting Maximalists will always have print, and this season encourages us to revisit the art of the clash with retro-inspired patterns. A devil-may-care approach will be rewarded here, but if your less-is-more tendencies are too strong to rebel against, a graphic top or skirt (ideally one that looks, or is, vintage) will do the trick. Denim with trims, per Valentino, can achieve the same mix-and-match effect. Crafty Minimalism Texture, fringe, and crochet can do much for a minimalist wardrobe with very little. The idea here isn’t to do away with the clean line entirely, but rather play up those refrained sensibilities with novelty and craft. Even purists will enjoy styling these pieces. Exaggerated Proportions Sculptural shapes alluding to a Japanese-style minimalism will inspire you to look differently at your everyday staples. These boisterous, asymmetric, and fanned-out silhouettes offer a true fashion POV to pieces you might wear everyday, from white T-shirts to nipped-in jackets. New Business Codes We’re leaving the office siren behind for spring, and embracing new tailoring codes for 2025. Simone Bellotti suggested a nipped-in peplum shape with drop shoulders at Bally, Stella McCartney went ’80s oversize, Tory Burch introduced a collarless, belted, and wrapped silhouette, and Blazy showed an almost cartoonishly oversize take at Bottega. A happy medium can be found at Saint Laurent, where the many full suiting looks in the collection were inspired by the house founder, and meant to convey “control and power,” as designer Anthony Vaccarello told Vogue’s Mark Holgate post-show backstage. Until these spring deliveries land, find inspiration for your own uniform refresh with current season styles from Saint Laurent, Tory Burch, Bally, and more."

In [4]:
ner = get_named_entities(content=text)

[{'text': 'vogue', 'type': 'ORG'},
 {'text': 'laird borelli-persson', 'type': 'PERSON'},
 {'text': 'proenza schouler', 'type': 'ORG'},
 {'text': 'yves stripe fringed knit dress', 'type': 'PRODUCT'},
 {'text': 'banana republic', 'type': 'ORG'},
 {'text': 'prada', 'type': 'ORG'},
 {'text': 'dries van noten', 'type': 'ORG'},
 {'text': 'simkhai', 'type': 'ORG'},
 {'text': 'calliope tailored vest', 'type': 'PRODUCT'},
 {'text': 'the frankie shop', 'type': 'ORG'},
 {'text': 'bea oversized cady blazer', 'type': 'PRODUCT'},
 {'text': 'diotima', 'type': 'ORG'},
 {'text': 'darliston skirt', 'type': 'PRODUCT'},
 {'text': 'alaïa', 'type': 'ORG'},
 {'text': 'lafayette 148 new york', 'type': 'ORG'},
 {'text': 'gesture handkerchief dress', 'type': 'PRODUCT'},
 {'text': 'aligne', 'type': 'ORG'},
 {'text': 'omari striped shorts', 'type': 'PRODUCT'},
 {'text': 'tory burch', 'type': 'ORG'},
 {'text': 'bottega', 'type': 'ORG'},
 {'text': 'gucci', 'type': 'ORG'},
 {'text': 'olympics', 'type': 'EVENT'},
 {'

In [7]:

# 1. Import from async_api instead of sync_api


No content was extracted.


In [2]:
import pandas as pd
from python.scripts.iab_taxonomy_converter import read_taxonomy_json


def generate_taxonomy_mapping(path=None):
    df = read_taxonomy_json(path=path)
    valid_mask = df['Unique ID 2'].notna() & (df['Unique ID 2'] != '')
    df_valid = df[valid_mask].copy()
    return pd.Series(df_valid['Unique ID'].values, index=df_valid['Unique ID 2']).to_dict()

In [3]:
path = '../data/raw/content_to_ad_product_taxonomy_mapping.tsv'
df = read_taxonomy_json(path=path)


In [19]:
uniqueIdValues = df['Unique ID'].tolist()
uniqueId2Values = df['Unique ID 2'].tolist()

mapping = {k: v for k, v in zip(uniqueId2Values, uniqueIdValues) if pd.notna(k) and k != ''}

mapping = dict()

for k, v in zip(uniqueIdValues, uniqueId2Values):
    if pd.notna(k) and k != '' and pd.notna(v) and v != '':
        if k not in mapping:
            mapping[k] = [v]
        else:
            mapping[k].append(v)



In [20]:
print(mapping)

{'1': ['1551'], '2': ['1551'], '3': ['1551'], '4': ['1551'], '5': ['1551'], '6': ['1551'], '7': ['1551'], '8': ['1551'], '9': ['1551'], '10': ['1551'], '11': ['1551'], '12': ['1551'], '13': ['1551'], '14': ['1551'], '15': ['1551'], '16': ['1551'], '17': ['1551'], '18': ['1551'], '19': ['1551'], '20': ['1551'], '21': ['1551'], '22': ['1551'], '23': ['1551'], '24': ['1551'], '25': ['1551'], '26': ['1551'], '27': ['1551'], '28': ['1551'], '29': ['1551'], '30': ['1551'], '31': ['1551'], '32': ['1558'], '33': ['1562'], '34': ['1562'], '35': ['1551'], '36': ['1562'], '37': ['1551'], '38': ['1551'], '39': ['1551'], '40': ['1551'], '41': ['1551'], '42': ['1422'], '43': ['1422'], '44': ['1422'], '45': ['1422'], '46': ['1422'], '47': ['1422'], '48': ['1422'], '49': ['1422'], '50': ['1422'], '51': ['1422'], '52': ['1010'], '53': ['1010'], '54': ['339'], '55': ['1016'], '56': ['1010'], '57': ['1019'], '58': ['1011'], '59': ['1010'], '60': ['1047'], '61': ['1010'], '62': ['1010'], '63': ['1010'], '

In [21]:

from typing import List, Dict, Any

from gliner import GLiNER

model = GLiNER.from_pretrained("urchade/gliner_small-v2.1")
entity_labels = [
    "product", "company", "technology",
    "industry", "location", "service", "topic",
    "person", "event", "organization"
]


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 62368.83it/s]
/Users/folarin/Workspace/contextual-ads-server/.venv/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [28]:
def extract_entities(text: str, threshold: float = 0.4) -> List[Dict]:
    """Extract named entities from text using GLiNER"""
    if not text:
        return []

    entities = model.predict_entities(
        text,
        entity_labels,
        threshold=threshold
    )

    # Deduplicate and normalize entities
    unique_entities = {}
    for ent in entities:
        key = (ent['text'].lower(), ent['label'])
        if key not in unique_entities or ent['score'] > unique_entities[key]['score']:
            unique_entities[key] = ent

    return list(unique_entities.values())

In [29]:
ans = extract_entities(text=text)

In [30]:
ans

[{'start': 8,
  'end': 26,
  'text': 'Spring/Summer 2025',
  'label': 'event',
  'score': 0.41004258394241333},
 {'start': 759,
  'end': 764,
  'text': 'Vogue',
  'label': 'organization',
  'score': 0.8451395034790039},
 {'start': 289,
  'end': 314,
  'text': 'spring/summer collections',
  'label': 'event',
  'score': 0.5758429765701294},
 {'start': 522,
  'end': 543,
  'text': 'Laird Borelli-Persson',
  'label': 'person',
  'score': 0.9306882619857788},
 {'start': 799,
  'end': 819,
  'text': 'The Summertime Dress',
  'label': 'product',
  'score': 0.4608513414859772},
 {'start': 821,
  'end': 837,
  'text': 'Proenza Schouler',
  'label': 'company',
  'score': 0.8251569271087646},
 {'start': 838,
  'end': 868,
  'text': 'Yves stripe fringed knit dress',
  'label': 'product',
  'score': 0.697951078414917},
 {'start': 877,
  'end': 896,
  'text': 'The Bohemian Blouse',
  'label': 'product',
  'score': 0.5753609538078308},
 {'start': 898,
  'end': 913,
  'text': 'Banana Republic',
  'lab

In [33]:
import spacy

nlp = spacy.load("en_core_web_sm")

text = text

doc_trf = nlp(text)

entities = dict()
for ent in doc_trf.ents:
    key = ent.text.lower()
    if key not in entities and ent.label_ in {"ORG", "PRODUCT", "PERSON", "EVENT", "GPE"}:
        entities[key] = (ent.text, ent.label_)
entities = list(entities.values())


In [34]:
entities

[('Vogue', 'PERSON'),
 ('Laird Borelli-Persson', 'PERSON'),
 ('The Bohemian Blouse', 'ORG'),
 ('Banana Republic', 'GPE'),
 ('Prada', 'ORG'),
 ('The Vintage-Inspired Print: Dries Van Noten', 'ORG'),
 ('The Frankie Shop Bea', 'ORG'),
 ('Diotima Darliston', 'PERSON'),
 ('The New Proportions Mini', 'ORG'),
 ('Alaïa', 'PERSON'),
 ('Aligne Omari', 'PERSON'),
 ('Tory Burch', 'PERSON'),
 ('Bottega', 'PERSON'),
 ('Suede', 'ORG'),
 ('Van Noten', 'PERSON'),
 ('Gucci', 'PERSON'),
 ('Miu Miu', 'PERSON'),
 ('grey, navy', 'ORG'),
 ('Spring Outerwear Stories', 'ORG'),
 ('Olympics', 'EVENT'),
 ('Sabato de Sarno', 'PERSON'),
 ('Matthieu Blazy', 'ORG'),
 ('Miuccia', 'PERSON'),
 ('Raf Simons', 'ORG'),
 ('Loewe', 'GPE'),
 ('Borelli-Persson', 'PERSON'),
 ('The Row’s ‘Tavishina', 'PRODUCT'),
 ('Blazy', 'ORG'),
 ('Tod', 'PERSON'),
 ('New York', 'GPE'),
 ('Daniella Kallmeyer', 'ORG'),
 ('Acne Studios', 'ORG'),
 ('Massimo Dutti', 'PERSON'),
 ('Ralph Lauren', 'PERSON'),
 ('Zara', 'PERSON'),
 ('Denim', 'PERSON'),

In [5]:
import asyncio
from playwright.async_api import async_playwright
from playwright_stealth import Stealth
from readability import Document
from bs4 import BeautifulSoup


async def extract_with_playwright(url, page_type="publisher"):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={'width': 1920, 'height': 1080},
            locale='en-US',
        )

        page = await context.new_page()

        # Apply stealth to avoid bot detection
        stealth = Stealth()
        await stealth.apply_stealth_async(page)

        try:
            await page.goto(url, wait_until="networkidle", timeout=45000)
            await page.wait_for_timeout(2000)
            content_html = await page.content()

        except Exception as e:
            print(f"Error fetching {url}: {e}")
            await browser.close()
            return None

        await browser.close()

    if page_type == "publisher":
        doc = Document(content_html)
        title = doc.title()
        clean_html = doc.summary()
        soup = BeautifulSoup(clean_html, 'lxml')
        text_content = soup.get_text(separator=' ', strip=True)
        return f"TITLE: {title}\n\nCONTENT: {text_content}"

    elif page_type == "landing_page":
        soup = BeautifulSoup(content_html, 'lxml')
        for tag in soup(['script', 'style', 'nav', 'footer', 'noscript']):
            tag.decompose()
        text_parts = []
        for tag in soup.find_all(['h1', 'h2', 'h3', 'p', 'li']):
            text = tag.get_text(strip=True)
            if len(text) > 10:
                text_parts.append(text)
        return "\n".join(text_parts)


In [6]:
url = "https://www.investors.com/news/tesla-stock-buy-zone-more-unsupervised-robotaxi-videos-elon-musk-court-win/"
page_type = "publisher"

# In a notebook you can use top-level await; this branch handles both notebook and script usage.
loop = asyncio.get_event_loop()
if loop.is_running():
    # Running inside an already-started event loop (Jupyter / PyCharm notebook)
    result = await extract_with_playwright(url, page_type)
else:
    # Running as a script where no loop is active
    result = asyncio.run(extract_with_playwright(url, page_type))

print(result)

Error fetching https://www.investors.com/news/tesla-stock-buy-zone-more-unsupervised-robotaxi-videos-elon-musk-court-win/: Page.goto: Timeout 45000ms exceeded.
Call log:
  - navigating to "https://www.investors.com/news/tesla-stock-buy-zone-more-unsupervised-robotaxi-videos-elon-musk-court-win/", waiting until "networkidle"

None


In [1]:
from transformers import pipeline

# Load a pre-trained NER pipeline
# By default, this uses a BERT-based model trained on CoNLL-2003
ner_tagger = pipeline("ner", aggregation_strategy="simple")

text = "Apple is looking at buying a startup in the United Kingdom for $1 billion."

results = ner_tagger(text)

for entity in results:
    print(f"Entity: {entity['word']} | Label: {entity['entity_group']} | Score: {entity['score']:.4f}")

/Users/folarin/Workspace/contextual-ads-server/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [8]:
from transformers import pipeline

ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

results = ner(text)

print(results)

for entity in results:
    print(f"{entity['word']}: {entity['entity_group']} (score: {entity['score']:.2f})")

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


[{'entity_group': 'MISC', 'score': np.float32(0.52098954), 'word': 'Can', 'start': 46, 'end': 49}, {'entity_group': 'ORG', 'score': np.float32(0.9924039), 'word': 'Vogue', 'start': 94, 'end': 99}, {'entity_group': 'ORG', 'score': np.float32(0.99596894), 'word': 'Vogue', 'start': 514, 'end': 519}, {'entity_group': 'PER', 'score': np.float32(0.9907969), 'word': 'Lai', 'start': 522, 'end': 525}, {'entity_group': 'PER', 'score': np.float32(0.9513188), 'word': '##rd Borelli', 'start': 525, 'end': 535}, {'entity_group': 'PER', 'score': np.float32(0.9899196), 'word': 'Persson', 'start': 536, 'end': 543}, {'entity_group': 'ORG', 'score': np.float32(0.9938559), 'word': 'Vogue', 'start': 759, 'end': 764}, {'entity_group': 'MISC', 'score': np.float32(0.8680235), 'word': '##time', 'start': 809, 'end': 813}, {'entity_group': 'PER', 'score': np.float32(0.9711265), 'word': 'Pro', 'start': 821, 'end': 824}, {'entity_group': 'PER', 'score': np.float32(0.392764), 'word': 'Sc', 'start': 829, 'end': 831},